# Dokumentasi Model Identifikasi Kesegaran Buah dan Sayur (Fresee)
**1. Tentang Model Ini**
<br> Dokumentasi ini berisi hasil training model yang disimpan dalam format file .h5.

**Informasi Model**
- Tujuan: Klasifikasi tingkat kesegaran buah dan sayuran.
- Format: .h5
- Kegunaan: Membantu sistem mengenali kondisi segar atau busuk pada komoditas yang diunggah.


**2. Unduh Model:**
<br>Anda dapat mengunduh file model (.h5) tersebut melalui tautan Google Drive di bawah ini:
https://drive.google.com/file/d/1as4PXO-pZfU8J7NegfAyncRnw9AU2DhR/view?usp=drive_link

**3. Cara Menggunakan Model**
<br>Jika Anda ingin menggunakan file ini dalam kode Anda, Anda hanya perlu menjalankan perintah sederhana berikut:

In [ ]:
from google.colab import files
from tensorflow.keras.models import load_model
import os
print("Silakan unggah file model (.h5) Anda:")
uploaded_model = files.upload()
model_filename = next(iter(uploaded_model.keys()))
model = load_model(model_filename)
print(f"Model '{model_filename}' berhasil dimuat dan siap digunakan!")

In [ ]:
from google.colab import files
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image
import time
def proses_dan_validasi_gambar(filename):
    try:
        img = Image.open(filename).convert('RGB')
        return img
    except FileNotFoundError:
        print(f"[ERROR] File gambar tidak ditemukan: {filename}")
        return None
    except Exception as e:
        print(f"[ERROR] Gagal memproses gambar {filename}: {e}")
        return None
print("Silakan unggah gambar (.png, .jpg, atau .jpeg)")
uploaded = files.upload()
if uploaded:
    filename = next(iter(uploaded.keys()))
    processed_img = proses_dan_validasi_gambar(filename)
    if processed_img is not None:
        img_resized = processed_img.resize((224, 224))
        img_array = img_to_array(img_resized) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        start_time = time.time()
        predictions = model.predict(img_array, verbose=0)
        end_time = time.time()
        inference_time = (end_time - start_time) * 1000
        print(f"Shape of predictions: {predictions.shape}")
        highest_prob_idx = np.argmax(predictions[0])
        confidence_score = predictions[0][highest_prob_idx] * 100
        class_labels = [
            'fresh_apples', 'fresh_banana', 'fresh_bittergroud', 'fresh_capsicum',
            'fresh_cucumber', 'fresh_okra', 'fresh_oranges', 'fresh_potato',
            'fresh_tomato', 'rotten_apples', 'rotten_banana', 'rotten_bittergroud',
            'rotten_capsicum', 'rotten_cucumber', 'rotten_okra', 'rotten_oranges',
            'rotten_potato', 'rotten_tomato'
        ]
        predicted_class_name = class_labels[highest_prob_idx]
        if 'fresh' in predicted_class_name:
            freshness_status = 'Segar'
            commodity_name = predicted_class_name.replace('fresh_', '').replace('_', ' ').capitalize()
        elif 'rotten' in predicted_class_name:
            freshness_status = 'Busuk (Tidak Segar)'
            commodity_name = predicted_class_name.replace('rotten_', '').replace('_', ' ').capitalize()
        else:
            freshness_status = 'Tidak Diketahui'
            commodity_name = predicted_class_name
        # ============================================================
        # MENAMPILKAN OUTPUT SESUAI PERMINTAAN
        # ============================================================
        plt.figure(figsize=(4, 4))
        plt.imshow(processed_img)
        plt.title(f"Hasil Identifikasi: {commodity_name}")
        plt.axis('off')
        plt.show()
        print("\n" + "="*40)
        print("          HASIL IDENTIFIKASI          ")
        print("="*40)
        print(f"Nama Komoditas      : {commodity_name}")
        print(f"Status Kesegaran    : {freshness_status}")
        print(f"Confidence Score    : {confidence_score:.2f}%")
        print(f"Inference Time      : {inference_time:.2f} ms")
        print("="*40)
    else:
        print("Gagal memproses gambar.")
else:
    print("Tidak ada file yang diunggah.")